# Model Fitting: Lag Selection and Lagged Multiple Regression

This notebook takes the cleaned quarterly datasets from preprocessing and:
1. Selects the time lag (0–4 quarters) at which each predictor is most strongly correlated with house price growth
2. Builds each model's dataset with predictors at their selected lags, then standardizes them
3. Fits an OLS multiple regression with Newey-West standard errors for the **long-run** and **recent** models
4. Saves results tables (coefficients, 95% CIs, p-values, R², adjusted R², VIF) for the evaluation step

**Input:** `DATA/cleaned_quarterly.csv`, produced by `SCRIPTS/preprocess_data.py`.

In [ ]:
# Import the libraries used in this notebook

import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Show numbers in tables with 4 decimal places
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Settings
The input is `DATA/cleaned_quarterly.csv` from `SCRIPTS/preprocess_data.py`. It already contains year-over-year changes, one row per calendar quarter. Quarters with missing values are kept as blank rows, so shifting by *k* rows always means *k* calendar quarters.

In [ ]:
# Settings (file locations, column names, analysis choices)

from pathlib import Path

# Find the project folder: the nearest folder (here or above) that contains DATA/.
# This works whether the notebook is run from the project folder or from SCRIPTS/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "DATA").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(f"No DATA folder found in {Path.cwd()} or any folder above it.")

DATA_PATH  = ROOT / "DATA" / "cleaned_quarterly.csv"
OUTPUT_DIR = ROOT / "OUTPUT"
print("Reading:", DATA_PATH)
print("Saving to:", OUTPUT_DIR)

# Column names in cleaned_quarterly.csv
DATE_COL = "quarter"        # e.g. "2017Q3"
OUTCOME  = "hpi_yoy_pct"    # YoY % change in the house price index

# Predictors used in each model
# Long-run model (1991 onward): only variables available back to 1990
LONG_RUN_PREDICTORS = ["unemployment_rate_yoy_pp", "mortgage_rate_yoy_pp"]
# Recent model (2017 onward): adds active listings, which only start in mid-2016
RECENT_PREDICTORS   = ["unemployment_rate_yoy_pp", "mortgage_rate_yoy_pp", "active_listings_yoy_pct"]

# Optional: limit the analysis window, e.g. END_QUARTER = "2025Q4". None = use all data.
START_QUARTER = None
END_QUARTER   = None

# Analysis parameters (from MI2)
MAX_LAG = 4      # test lags 0, 1, 2, 3, 4 quarters
NW_LAGS = 4      # Newey-West maximum lag
ALPHA   = 0.05   # two-sided significance level

# Create the OUTPUT folder if it doesn't exist yet
os.makedirs(OUTPUT_DIR, exist_ok=True)

Reading: /Users/haero/DS4002/Project2/DS4002-Project2/DATA/cleaned_quarterly.csv
Saving to: /Users/haero/DS4002/Project2/DS4002-Project2/OUTPUT


## 2. Load and check the data
Loads the cleaned file onto a continuous quarterly calendar. Missing rows are **not** dropped yet: dropping them before lagging would make a shift of *k* rows skip over gaps, so lags would no longer equal *k* calendar quarters. Missing values are removed later, after lagging. Each model then automatically uses only the quarters where all of its variables exist: the long-run model starts where unemployment begins, and the recent model where listings begin.

In [ ]:
# Load the cleaned data and define each model's time window

# Read the CSV and use the quarter (e.g. 2017Q3) as the row label
df = pd.read_csv(DATA_PATH)
df.index = pd.PeriodIndex(df.pop(DATE_COL).astype(str), freq="Q")
df = df.sort_index()

# Make sure every column we need exists; stop with a clear message if not
needed = [OUTCOME, *dict.fromkeys(LONG_RUN_PREDICTORS + RECENT_PREDICTORS)]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Columns {missing} not found. Columns are: {list(df.columns)}")

# Guarantee one row per calendar quarter (blank rows kept), then apply the window
df = df[needed].reindex(pd.period_range(df.index.min(), df.index.max(), freq="Q"))
df = df.loc[START_QUARTER:END_QUARTER]

# Split the columns needed for each model
long_run = df[[OUTCOME, *LONG_RUN_PREDICTORS]]
recent   = df[[OUTCOME, *RECENT_PREDICTORS]]

# Each model's window: first to last quarter where all its variables are present.
# Predictors are shifted using the full series, so lagged values can come from just
# before the window, but every correlation and regression uses only window quarters.
def window(d):
    complete = d.dropna()
    return complete.index.min(), complete.index.max()

long_window, recent_window = window(long_run), window(recent)
# Print each window and how many complete quarters it has
for name, d, (start, end) in [("Long-run", long_run, long_window), ("Recent", recent, recent_window)]:
    n = len(d.loc[start:end].dropna())
    print(f"{name} window: {start} to {end} ({n} complete quarters before lagging)")
print()
print("Missing values per column:"); print(df.isna().sum())

Long-run window: 1991Q1 to 2026Q2 (141 complete quarters before lagging)
Recent window: 2017Q3 to 2026Q2 (35 complete quarters before lagging)

Missing values per column:
hpi_yoy_pct                   4
unemployment_rate_yoy_pp     31
mortgage_rate_yoy_pp          0
active_listings_yoy_pct     136
dtype: int64


## 3. Lag selection
For each predictor, compute the Pearson correlation between house price growth this quarter and the predictor's change *k* quarters earlier, for k = 0–4. The lag with the **largest absolute correlation** is selected. All five correlations are reported, as the MI2 requires.

> Because lags are chosen using the same data the model is fit on, treat the resulting p-values as exploratory (as stated in the MI2).

All five lags are compared on the same set of quarters, so the differences come from the lag itself and not from different samples.

In [ ]:
# Lag selection — find how long each factor takes to show up in prices

def lagged_frame(df, p, win):
    # Outcome plus predictor p at lags 0..MAX_LAG, on ONE common set of quarters,
    # so every lag is compared on exactly the same data.
    X = pd.concat({k: df[p].shift(k) for k in range(MAX_LAG + 1)}, axis=1)
    return pd.concat([df[OUTCOME].rename("y"), X], axis=1).loc[win[0]:win[1]].dropna()


def lag_correlations(df, predictors, model_name, win):
    rows = []
    for p in predictors:
        d = lagged_frame(df, p, win)
        for k in range(MAX_LAG + 1):
            rows.append({"model": model_name, "predictor": p, "lag_quarters": k,
                         "correlation": d["y"].corr(d[k]), "n_quarters": len(d)})
    table = pd.DataFrame(rows)
    best = table.loc[table.groupby("predictor")["correlation"].apply(lambda s: s.abs().idxmax())]
    table["selected"] = table.index.isin(best.index)
    selected_lags = dict(zip(best["predictor"], best["lag_quarters"]))
    return table, selected_lags

# Run lag selection separately for each model (each uses its own time window)
long_corr, long_lags     = lag_correlations(long_run, LONG_RUN_PREDICTORS, "long_run", long_window)
recent_corr, recent_lags = lag_correlations(recent,   RECENT_PREDICTORS,   "recent",   recent_window)

print("Selected lags (quarters), long-run:", long_lags)
print("Selected lags (quarters), recent:  ", recent_lags)

# Show all five correlations for every predictor (rows) and lag (columns)
lag_table = pd.concat([long_corr, recent_corr], ignore_index=True)
lag_table.pivot_table(index=["model", "predictor"], columns="lag_quarters", values="correlation")

Selected lags (quarters), long-run: {'mortgage_rate_yoy_pp': 0, 'unemployment_rate_yoy_pp': 1}
Selected lags (quarters), recent:   {'active_listings_yoy_pct': 4, 'mortgage_rate_yoy_pp': 0, 'unemployment_rate_yoy_pp': 1}


lag_quarters                            0       1       2       3       4
model    predictor                                                       
long_run mortgage_rate_yoy_pp      0.3249  0.2769  0.2187  0.1942  0.1450
         unemployment_rate_yoy_pp -0.1936 -0.2232 -0.2090 -0.1550 -0.0767
recent   active_listings_yoy_pct  -0.0982 -0.3662 -0.5154 -0.6324 -0.7045
         mortgage_rate_yoy_pp      0.6733  0.4234  0.1900  0.0466 -0.1078
         unemployment_rate_yoy_pp -0.4302 -0.4935 -0.4510 -0.3450 -0.1502

## 3b. How reliable is each selected lag?
Year-over-year changes overlap: two consecutive values share 3 of their 4 quarters. So a predictor's correlations at neighboring lags are naturally similar, and the "best" lag can win by a tiny margin. To measure this, we use a **moving block bootstrap**: we resample the data in blocks of 8 consecutive quarters (which keeps the overlap structure intact), repeat the lag selection 2,000 times, and record how often each lag wins.

**How to read it:** if one lag wins in most resamples (e.g., 70% or more), the response time is reasonably clear. If wins are spread across several lags, report a **range** of lags (e.g., "0 to 2 quarters") rather than a single number. The regression's standard errors already account for the overlap through the Newey-West correction (maximum lag 4 covers the 3-quarter overlap).

In [ ]:
# How reliable is each selected lag? (moving block bootstrap)

N_BOOT = 2000   # number of resamples
BLOCK  = 8      # block length in quarters (2 years)
SEED   = 0      # fixed random seed so results are exactly reproducible
rng = np.random.default_rng(SEED)

def lag_stability(df, predictors, model_name, win):
    rows = []
    for p in predictors:
        d = lagged_frame(df, p, win)
        y, Z, n = d["y"].values, d[list(range(MAX_LAG + 1))].values, len(d)
        wins = np.zeros(MAX_LAG + 1)
        for _ in range(N_BOOT):
            starts = rng.integers(0, n, int(np.ceil(n / BLOCK)))
            idx = np.concatenate([np.arange(s, s + BLOCK) % n for s in starts])[:n]
            r = [np.corrcoef(y[idx], Z[idx, k])[0, 1] for k in range(MAX_LAG + 1)]
            wins[int(np.nanargmax(np.abs(r)))] += 1
        row = {"model": model_name, "predictor": p}
        row.update({f"lag{k}_%": 100 * wins[k] / N_BOOT for k in range(MAX_LAG + 1)})
        rows.append(row)
    return pd.DataFrame(rows)

stability = pd.concat([lag_stability(long_run, LONG_RUN_PREDICTORS, "long_run", long_window),
                       lag_stability(recent,   RECENT_PREDICTORS,   "recent",   recent_window)],
                      ignore_index=True)

# Add the selected lag and how often it won; call it "clear" if it won in at least 70% of resamples
stability["selected_lag"] = [ (long_lags if m == "long_run" else recent_lags)[p]
                              for m, p in zip(stability.model, stability.predictor)]
stability["selected_lag_win_%"] = [r[f"lag{r.selected_lag}_%"] for _, r in stability.iterrows()]
stability["lag_is_clear"] = stability["selected_lag_win_%"] >= 70
stability.round(1)

,model,predictor,lag0_%,lag1_%,lag2_%,lag3_%,lag4_%,selected_lag,selected_lag_win_%,lag_is_clear
0,long_run,unemployment_rate_yoy_pp,18.2000,37.4000,18.2000,9.4000,16.8000,1,37.4000,False
1,long_run,mortgage_rate_yoy_pp,72.5000,6.2000,1.4000,10.7000,9.2000,0,72.5000,True
2,recent,unemployment_rate_yoy_pp,7.2000,51.4000,10.6000,11.0000,19.8000,1,51.4000,False
3,recent,mortgage_rate_yoy_pp,86.2000,2.2000,2.4000,4.9000,4.2000,0,86.2000,True
4,recent,active_listings_yoy_pct,6.0000,3.1000,3.0000,8.4000,79.5000,4,79.5000,True


## 4. Build each model's dataset
Shift each predictor by its selected lag, drop the quarters lost to lagging, then **standardize** the predictors (mean 0, standard deviation 1, computed on the final model sample) so their coefficients can be compared and ranked. The outcome stays in percentage-point units of price growth.

In [ ]:
# Build each model's dataset (predictors at their selected lags, standardized)

def build_model_data(df, selected_lags, win):
    # Shift each predictor by its selected lag; name columns like "mortgage_rate_yoy_pp_lag0"
    X = pd.DataFrame({f"{p}_lag{k}": df[p].shift(k) for p, k in selected_lags.items()})
    # Keep only quarters inside the model's window where every value is present
    data = pd.concat([df[OUTCOME], X], axis=1).loc[win[0]:win[1]].dropna()
    y = data[OUTCOME]
    X_raw = data.drop(columns=OUTCOME)
    X_std = (X_raw - X_raw.mean()) / X_raw.std()
    return y, X_std, X_raw


y_long, X_long, X_long_raw       = build_model_data(long_run, long_lags, long_window)
y_recent, X_recent, X_recent_raw = build_model_data(recent, recent_lags, recent_window)

print(f"Long-run model sample: {y_long.index.min()} to {y_long.index.max()}, n = {len(y_long)}")
print(f"Recent model sample:   {y_recent.index.min()} to {y_recent.index.max()}, n = {len(y_recent)}")

Long-run model sample: 1991Q2 to 2026Q2, n = 140
Recent model sample:   2018Q3 to 2026Q2, n = 31


## 5. Fit the regressions
Ordinary least squares with an intercept, using Newey-West (HAC) standard errors with a maximum lag of 4 quarters. `use_t=True` makes the coefficient tests two-sided **t-tests**, matching the MI2.

In [ ]:
# Fit the regression models (OLS with Newey-West standard errors)

def fit_model(y, X):
    # OLS regression with an intercept (add_constant).
    # cov_type="HAC" = Newey-West standard errors, which correct for autocorrelation.
    # use_t=True makes the coefficient tests t-tests (two-sided), as planned in the MI2.
    return sm.OLS(y, sm.add_constant(X)).fit(
        cov_type="HAC", cov_kwds={"maxlags": NW_LAGS}, use_t=True)


long_model   = fit_model(y_long, X_long)
recent_model = fit_model(y_recent, X_recent)

# Full regression output for the long-run model
print(long_model.summary(title="Long-run model: YoY house price growth"))

                    Long-run model: YoY house price growth                    
Dep. Variable:            hpi_yoy_pct   R-squared:                       0.133
Model:                            OLS   Adj. R-squared:                  0.121
Method:                 Least Squares   F-statistic:                     3.404
Date:                Tue, 20 Oct 2026   Prob (F-statistic):             0.0361
Time:                        18:42:19   Log-Likelihood:                -423.92
No. Observations:                 140   AIC:                             853.8
Df Residuals:                     137   BIC:                             862.7
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const         

In [ ]:
# Full regression output for the recent model
print(recent_model.summary(title="Recent model: YoY house price growth"))

                     Recent model: YoY house price growth                     
Dep. Variable:            hpi_yoy_pct   R-squared:                       0.692
Model:                            OLS   Adj. R-squared:                  0.658
Method:                 Least Squares   F-statistic:                     31.76
Date:                Tue, 20 Oct 2026   Prob (F-statistic):           5.29e-09
Time:                        18:42:45   Log-Likelihood:                -73.215
No. Observations:                  31   AIC:                             154.4
Df Residuals:                      27   BIC:                             160.2
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const         

## 6. Results tables
One tidy table per model with the coefficient, 95% confidence interval, p-value, significance flag, rank by |standardized coefficient|, and VIF, plus a model-level summary. These match the MI2 quantifiable goal.

In [ ]:
# Results tables (coefficients, 95% CIs, p-values, ranks, VIF, R²)

def results_table(model, X, model_name, selected_lags):
    ci = model.conf_int(alpha=ALPHA)                      # 95% confidence intervals
    t = pd.DataFrame({
        "model": model_name,
        "term": model.params.index,                       # "const" = intercept
        "std_coefficient": model.params.values,           # effect of a 1-SD increase in the predictor
        "std_error_NW": model.bse.values,                 # Newey-West standard error
        "ci_95_low": ci[0].values,
        "ci_95_high": ci[1].values,
        "p_value": model.pvalues.values,
    })
    t["significant"] = t["p_value"] < ALPHA

    # Rank predictors (not the intercept) by size of standardized coefficient: 1 = strongest
    preds = t["term"] != "const"
    t.loc[preds, "rank"] = t.loc[preds, "std_coefficient"].abs().rank(ascending=False).astype(int)

    # Variance inflation factor for each predictor (above 5 = too much overlap between predictors)
    exog = sm.add_constant(X).values
    vif = {col: variance_inflation_factor(exog, i + 1) for i, col in enumerate(X.columns)}
    t["VIF"] = t["term"].map(vif)
    return t


def model_summary(model, name):
    # One row of model-level results
    return {"model": name, "n_quarters": int(model.nobs), "R2": model.rsquared,
            "adj_R2": model.rsquared_adj,
            "n_significant_predictors": int((model.pvalues.drop("const") < ALPHA).sum())}


results = pd.concat([
    results_table(long_model, X_long, "long_run", long_lags),
    results_table(recent_model, X_recent, "recent", recent_lags),
], ignore_index=True)

summary = pd.DataFrame([model_summary(long_model, "long_run"),
                        model_summary(recent_model, "recent")])
display(summary)   # model-level results (R², adjusted R², sample size)
results            # predictor-level results

,model,n_quarters,R2,adj_R2,n_significant_predictors
0,long_run,140,0.1334,0.1207,1
1,recent,31,0.6925,0.6583,3


,model,term,std_coefficient,std_error_NW,ci_95_low,ci_95_high,p_value,significant,rank,VIF
0,long_run,const,4.4943,0.8775,2.7591,6.2294,0.0000,True,NaN,NaN
1,long_run,mortgage_rate_yoy_pp_lag0,1.5638,0.6409,0.2965,2.8311,0.0160,True,1.0000,1.0442
2,long_run,unemployment_rate_yoy_pp_lag1,-0.9156,0.6199,-2.1414,0.3102,0.1420,False,2.0000,1.0442
3,recent,const,7.0174,0.6919,5.5978,8.4369,0.0000,True,NaN,NaN
4,recent,active_listings_yoy_pct_lag4,-2.1070,0.8321,-3.8142,-0.3998,0.0175,True,1.0000,1.3245
5,recent,mortgage_rate_yoy_pp_lag0,1.8991,0.3436,1.1941,2.6041,0.0000,True,2.0000,1.3122
6,recent,unemployment_rate_yoy_pp_lag1,-1.0064,0.3834,-1.7930,-0.2198,0.0141,True,3.0000,1.1742


## 7. Save outputs for the evaluation step
These files are what Hershi uses for significance testing, diagnostics, and figures.

In [ ]:
# Save all results to the OUTPUT folder (used by the evaluation step)

lag_table.to_csv(OUTPUT_DIR / "lag_correlations.csv", index=False)   # all lag correlations
stability.to_csv(OUTPUT_DIR / "lag_stability.csv", index=False)      # lag reliability check
results.to_csv(OUTPUT_DIR / "results_table.csv", index=False)        # coefficients, p-values, VIF
summary.to_csv(OUTPUT_DIR / "model_summary.csv", index=False)        # R², adjusted R², n

# Full text summaries of both regressions
with open(OUTPUT_DIR / "long_run_model_summary.txt", "w") as f:
    f.write(long_model.summary().as_text())
with open(OUTPUT_DIR / "recent_model_summary.txt", "w") as f:
    f.write(recent_model.summary().as_text())

# Actual values, fitted values, and residuals for each quarter (for residual plots)
for name, m, y in [("long_run", long_model, y_long), ("recent", recent_model, y_recent)]:
    pd.DataFrame({"quarter": y.index.astype(str), "actual": y.values,
                  "fitted": m.fittedvalues.values, "residual": m.resid.values}
                 ).to_csv(OUTPUT_DIR / f"{name}_fitted_residuals.csv", index=False)

print("Saved to", OUTPUT_DIR, ":", sorted(os.listdir(OUTPUT_DIR)))

Saved to /Users/haero/DS4002/Project2/DS4002-Project2/OUTPUT : ['hpi_growth_distribution.png', 'lag_correlations.csv', 'lag_stability.csv', 'long_run_fitted_residuals.csv', 'long_run_model_summary.txt', 'model_summary.csv', 'quarterly_trends.png', 'recent_fitted_residuals.csv', 'recent_model_summary.txt', 'results_table.csv']


## 8. Interpreting the output
- **std_coefficient:** the change in YoY house price growth (percentage points) associated with a **one standard deviation** increase in that predictor, holding the others constant. A negative value means the predictor is associated with slower price growth.
- **rank:** predictors ordered by the size of their standardized coefficient, so 1 is the strongest association.
- **selected lag:** the number in each term name (e.g., `mortgage_rate_lag3`) is how many quarters earlier the predictor is measured. This is the "how quickly do prices respond" answer.
- **p_value / significant:** a two-sided test at α = 0.05 with Newey-West standard errors. These are exploratory, because lags were chosen using the same data.
- **VIF above 5:** predictors overlap too much to separate their individual associations reliably.
- **R² / adj_R2:** the share of the quarter-to-quarter variation in price growth the model explains. Use adjusted R² to compare the two models, since they have different numbers of predictors.